# Multi-Agent Fraud Detection Architecture

```text
                           ┌─────────────────────┐
                           │   Dataset Loader    │
                           │ tx/users/geo/sms/   │
                           │ mails/audio         │
                           └─────────┬───────────┘
                                     │
                           ┌─────────▼───────────┐
                           │ Context Builder     │
                           │ user memory/state   │
                           │ feature cache       │
                           └─────────┬───────────┘
                                     │
        ┌────────────────────────────┼─────────────────────────────┐
        │                            │                             │
┌───────▼────────┐         ┌────────▼────────┐          ┌─────────▼────────┐
│ Behavior Agent │         │ Geo-Time Agent  │          │ Comm Risk Agent  │
│ anomalies      │         │ movement, time  │          │ sms/email        │
└───────┬────────┘         └────────┬────────┘          └─────────┬────────┘
        │                            │                             │
        │                    ┌───────▼────────┐           ┌───────▼────────┐
        │                    │ Audio Risk      │           │ Drift Agent     │
        │                    │ transcription    │           │ new fraud modes │
        │                    │ + scam cues      │           └───────┬────────┘
        │                    └───────┬────────┘                   │
        └────────────────────────────┼─────────────────────────────┘
                                     │
                           ┌─────────▼───────────┐
                           │ Risk Fusion Agent   │
                           │ calibrated score    │
                           └─────────┬───────────┘
                                     │
                         low/high confidence?
                           │                    │
                         yes                    no
                           │                    │
                  ┌────────▼───────┐   ┌────────▼────────┐
                  │ Final Decision │   │ LLM Judge Agent │
                  │ legit/fraud    │   │ only on hard    │
                  └────────┬───────┘   │ expensive cases │
                           │            └────────┬────────┘
                           └─────────────────────┘
                                     │
                           ┌─────────▼───────────┐
                           │ Submission Writer   │
                           │ fraudulent IDs.txt  │
                           └─────────────────────┘

In [16]:
import os
import re
import json
import math
import unicodedata
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Any, Tuple

import numpy as np
import pandas as pd
from dotenv import load_dotenv
import tqdm as tqdm
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langfuse import Langfuse, observe
from langfuse.langchain import CallbackHandler




In [2]:
# 0. CONFIG

load_dotenv()

TEAM_NAME = os.getenv("TEAM_NAME", "MuNA")
DATA_DIR = os.getenv("DATA_DIR", "../dataset")
OUTPUT_PATH = os.getenv("OUTPUT_PATH", "../fraudulent_transactions.txt")
DEBUG_CSV = os.getenv("DEBUG_CSV", "../debug_scored_transactions.csv")

# Fast execution controls
USE_LLM = os.getenv("USE_LLM", "0") == "1"          # default OFF for speed
MAX_LLM_ESCALATIONS = int(os.getenv("MAX_LLM_ESCALATIONS", "25"))
USE_AUDIO_LLM = os.getenv("USE_AUDIO_LLM", "0") == "1"
WHISPER_MODEL_SIZE = os.getenv("WHISPER_MODEL_SIZE", "tiny")  # tiny/base/small

llm = ChatOpenAI(
    model="openai/gpt-4.1-mini",
    temperature=0,
    max_retries=1,
    timeout=20,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENAI_API_KEY"),
    default_headers={
        "HTTP-Referer": "http://localhost",
        "X-Title": "Fraud Detection Agent",
    },
)

langfuse_client = Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host=os.getenv("LANGFUSE_HOST", "https://challenges.reply.com/langfuse"),
)

def generate_session_id() -> str:
    import ulid
    return f"{TEAM_NAME.replace(' ', '-')}-{ulid.new().str}"

In [3]:
# 1. HELPERS

def normalize_text(s: Any) -> str:
    s = "" if s is None else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", s.strip().lower())


def safe_float(x: Any, default: float = 0.0) -> float:
    try:
        return float(x)
    except Exception:
        return default


def sigmoid(x: float) -> float:
    return 1 / (1 + math.exp(-x))


def log1p_clip(x: float) -> float:
    return math.log1p(max(x, 0.0))


In [4]:
# 2. DATA LOADER

@dataclass
class DatasetBundle:
    transactions: pd.DataFrame
    users: List[Dict[str, Any]]
    locations: List[Dict[str, Any]]
    sms: List[Dict[str, Any]]
    mails: List[Dict[str, Any]]
    audio_files: List[Path]


class DataLoader:

    @staticmethod
    def load_single_bundle(dataset_path: Path) -> DatasetBundle:
        """
        Load one dataset folder
        """
        tx = pd.read_csv(dataset_path / "transactions.csv")
        tx.columns = [normalize_text(c).replace(" ", "_") for c in tx.columns]

        rename_map = {
            "transaction_id": "transaction_id",
            "sender_id": "sender_id",
            "recipient_id": "recipient_id",
            "transaction_type": "transaction_type",
            "amount": "amount",
            "location": "location",
            "payment_method": "payment_method",
            "sender_iban": "sender_iban",
            "recipient_iban": "recipient_iban",
            "balance": "balance_after",
            "timestamp": "timestamp",
            "description": "description",
        }

        tx = tx.rename(columns={k: v for k, v in rename_map.items() if k in tx.columns})

        for col in [
            "sender_id", "recipient_id", "transaction_type", "location",
            "payment_method", "sender_iban", "recipient_iban", "description"
        ]:
            if col not in tx.columns:
                tx[col] = ""
            tx[col] = tx[col].fillna("")

        tx["amount"] = pd.to_numeric(tx["amount"], errors="coerce").fillna(0.0)
        tx["balance_after"] = pd.to_numeric(tx["balance_after"], errors="coerce").fillna(0.0)
        tx["timestamp"] = pd.to_datetime(tx["timestamp"], errors="coerce")

        def load_json(file):
            path = dataset_path / file
            if path.exists():
                with open(path, "r", encoding="utf-8") as f:
                    return json.load(f)
            return []

        users = load_json("users.json")
        locations = load_json("locations.json")
        sms = load_json("sms.json")
        mails = load_json("mails.json")

        audio_dir = dataset_path / "audio"
        audio_files = sorted(audio_dir.glob("*.mp3")) if audio_dir.exists() else []

        return DatasetBundle(
            transactions=tx,
            users=users,
            locations=locations,
            sms=sms,
            mails=mails,
            audio_files=audio_files,
        )

    @staticmethod
    def load_all_bundles(data_dir: str) -> Dict[str, DatasetBundle]:
        """
        Load all dataset folders inside DATA_DIR
        """
        base = Path(data_dir)
        bundles = {}

        for dataset_folder in base.iterdir():
            if dataset_folder.is_dir():
                print(f"Loading dataset: {dataset_folder.name}")
                bundle = DataLoader.load_single_bundle(dataset_folder)
                bundles[dataset_folder.name] = bundle

        return bundles

In [5]:
# 3. AGENT OUTPUT

@dataclass
class AgentResult:
    score: float
    reasons: List[str] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)


In [6]:
# 4. CONTEXT BUILDER AGENT


class ContextBuilderAgent:
    def __init__(self, bundle: DatasetBundle):
        self.bundle = bundle
        self.tx = bundle.transactions.sort_values("timestamp").copy()

        self.location_df = pd.DataFrame(bundle.locations)
        if not self.location_df.empty:
            self.location_df.columns = [normalize_text(c).replace(" ", "_") for c in self.location_df.columns]
            if "timestamp" in self.location_df.columns:
                self.location_df["timestamp"] = pd.to_datetime(self.location_df["timestamp"], errors="coerce")

        self.users_df = pd.DataFrame(bundle.users)
        if not self.users_df.empty:
            self.users_df["full_name_norm"] = (
                self.users_df.get("first_name", "").astype(str).map(normalize_text) + " " +
                self.users_df.get("last_name", "").astype(str).map(normalize_text)
            ).str.strip()
            self.users_df["iban_norm"] = self.users_df.get("iban", "").astype(str).map(normalize_text)

        self.sender_stats = self.tx.groupby("sender_id").agg(
            amount_mean=("amount", "mean"),
            amount_std=("amount", "std"),
            amount_median=("amount", "median"),
            tx_count=("transaction_id", "count"),
        ).fillna(0)

        self.audio_index = self._build_audio_index(bundle.audio_files)

    def _build_audio_index(self, audio_files: List[Path]) -> pd.DataFrame:
        rows = []
        for p in audio_files:
            m = re.match(r"(\d{8})_(\d{6})-(.+)", p.stem)
            ts = None
            person_key = ""
            if m:
                date_part, time_part, person = m.groups()
                ts = pd.to_datetime(date_part + time_part, format="%Y%m%d%H%M%S", errors="coerce")
                person_key = normalize_text(person.replace("_", " "))
            rows.append({
                "path": str(p),
                "timestamp": ts,
                "person_key": person_key
            })
        return pd.DataFrame(rows)

    def get_sender_stats(self, sender_id: str) -> Dict[str, float]:
        if sender_id in self.sender_stats.index:
            return self.sender_stats.loc[sender_id].to_dict()
        return {"amount_mean": 0.0, "amount_std": 0.0, "amount_median": 0.0, "tx_count": 0.0}

    def recent_transactions(self, sender_id: str, ts: pd.Timestamp, n: int = 30) -> pd.DataFrame:
        hist = self.tx[(self.tx["sender_id"] == sender_id) & (self.tx["timestamp"] < ts)]
        return hist.tail(n)

    def recent_locations(self, ts: pd.Timestamp, days: int = 7) -> pd.DataFrame:
        if self.location_df.empty or "timestamp" not in self.location_df.columns:
            return pd.DataFrame()
        lo = ts - pd.Timedelta(days=days)
        hi = ts + pd.Timedelta(hours=6)
        return self.location_df[(self.location_df["timestamp"] >= lo) & (self.location_df["timestamp"] <= hi)]

    def candidate_messages(self, row: pd.Series) -> Dict[str, List[str]]:
        desc = normalize_text(row.get("description", ""))
        loc = normalize_text(row.get("location", ""))

        sms_hits = []
        for item in self.bundle.sms:
            txt = item.get("sms", "")
            low = normalize_text(txt)
            if any(k in low for k in [desc[:20], loc, "urgent", "verify", "security", "customs", "crypto"] if k):
                sms_hits.append(txt)

        mail_hits = []
        for item in self.bundle.mails:
            txt = item.get("mail", "")
            low = normalize_text(txt)
            if any(k in low for k in [desc[:20], loc, "urgent", "verify", "security", "invoice", "payment"] if k):
                mail_hits.append(txt)

        return {"sms": sms_hits[:6], "mails": mail_hits[:6]}

    def candidate_audio(self, row: pd.Series) -> List[str]:
        if self.audio_index.empty:
            return []

        # Weak match: most recent audio before transaction
        ts = row["timestamp"]
        if pd.isna(ts):
            return []

        df = self.audio_index[self.audio_index["timestamp"] <= ts].copy()
        df = df.sort_values("timestamp", ascending=False)
        return df["path"].head(3).tolist()


In [7]:
# 5. BEHAVIOR AGENT

class BehaviorAgent:
    @observe()
    def run(self, row: pd.Series, ctx: ContextBuilderAgent) -> AgentResult:
        sender_id = row["sender_id"]
        amount = safe_float(row["amount"])
        ts = row["timestamp"]

        reasons = []
        score = 0.0

        stats = ctx.get_sender_stats(sender_id)
        mean_amt = safe_float(stats["amount_mean"])
        std_amt = safe_float(stats["amount_std"], 0.0)
        std_amt = std_amt if std_amt > 1e-6 else max(mean_amt * 0.30, 1.0)

        z = abs(amount - mean_amt) / std_amt if mean_amt > 0 else 0.0
        if z > 3:
            score += 0.35
            reasons.append(f"amount anomaly z={z:.2f}")

        hist = ctx.recent_transactions(sender_id, ts, n=30)
        if not hist.empty:
            if row["recipient_id"] and row["recipient_id"] not in set(hist["recipient_id"].astype(str)):
                score += 0.15
                reasons.append("new recipient")

            if row["payment_method"] and row["payment_method"] not in set(hist["payment_method"].astype(str)):
                score += 0.10
                reasons.append("new payment method")

            if row["transaction_type"] not in set(hist["transaction_type"].astype(str)):
                score += 0.10
                reasons.append("new transaction type")

            last_24h = hist[hist["timestamp"] >= ts - pd.Timedelta(hours=24)]
            if len(last_24h) >= 5:
                score += 0.20
                reasons.append("velocity spike")

        if pd.notna(ts) and (ts.hour <= 4 or ts.hour >= 23):
            score += 0.10
            reasons.append("odd hour")

        if safe_float(row.get("balance_after", 0)) < 0:
            score += 0.20
            reasons.append("negative balance after")

        return AgentResult(min(score, 1.0), reasons, {"amount_z": round(z, 3)})

In [8]:
# 6. GEO-TIME AGENT

class GeoTimeAgent:
    @observe()
    def run(self, row: pd.Series, ctx: ContextBuilderAgent) -> AgentResult:
        reasons = []
        score = 0.0

        tx_loc = normalize_text(row.get("location", ""))
        ts = row["timestamp"]

        if tx_loc:
            recent_geo = ctx.recent_locations(ts, days=7)
            if not recent_geo.empty and "city" in recent_geo.columns:
                recent_cities = set(recent_geo["city"].astype(str).map(normalize_text))
                if recent_cities and not any(city in tx_loc for city in recent_cities):
                    score += 0.25
                    reasons.append("location mismatch vs recent geo")

            hist = ctx.recent_transactions(row["sender_id"], ts, n=30)
            prior_locs = set(hist["location"].astype(str).map(normalize_text)) - {""}
            if prior_locs and tx_loc not in prior_locs:
                score += 0.15
                reasons.append("new in-person location")

        if pd.notna(ts) and tx_loc and ts.hour <= 5:
            score += 0.10
            reasons.append("in-person payment at unusual hour")

        return AgentResult(min(score, 1.0), reasons, {})

In [9]:
# 7. COMMUNICATION AGENT

class CommunicationAgent:
    SUSPICIOUS = [
        "urgent", "verify", "security alert", "account lock", "suspicious login",
        "customs", "pay now", "otp", "password", "wallet", "crypto",
        "refund", "gift card", "bit.ly", "http://", "amaz0n", "paypa1"
    ]

    def __init__(self):
        self.cache: Dict[str, AgentResult] = {}

    @observe()
    def cheap_run(self, messages: Dict[str, List[str]]) -> AgentResult:
        text = "\n".join(messages.get("sms", []) + messages.get("mails", [])).lower()
        hits = [p for p in self.SUSPICIOUS if p in text]
        score = min(len(hits) * 0.08, 0.75)
        return AgentResult(score, hits[:6], {"pattern_hits": len(hits)})

    @observe()
    def llm_run(self, messages: Dict[str, List[str]], session_id: str) -> AgentResult:
        cheap = self.cheap_run(messages)
        if not USE_LLM or cheap.score < 0.20:
            return cheap

        sms = messages.get("sms", [])[:3]
        mails = messages.get("mails", [])[:3]
        cache_key = json.dumps({"sms": sms, "mails": mails}, ensure_ascii=False, sort_keys=True)

        if cache_key in self.cache:
            return self.cache[cache_key]

        prompt = f"""
You are a fraud communications analyst.

Analyze the following SMS and email snippets for phishing, scam, coercion,
account takeover, or social engineering risk.

Return strict JSON:
{{
  "risk": 0.0,
  "category": "benign|phishing|takeover|scam|mixed",
  "reasons": ["short reason 1", "short reason 2"]
}}

SMS:
{sms}

EMAILS:
{mails}
""".strip()

        try:
            handler = CallbackHandler()
            resp = llm.invoke(
                [HumanMessage(content=prompt)],
                config={
                    "callbacks": [handler],
                    "metadata": {
                        "langfuse_session_id": session_id,
                        "agent_name": "CommunicationAgent"
                    }
                }
            )
            text = resp.content if hasattr(resp, "content") else str(resp)
            data = json.loads(text)
            risk = float(data.get("risk", 0.35))
            reasons = data.get("reasons", ["llm comm risk"])
            result = AgentResult(min(max(risk, 0.0), 1.0), reasons[:6], {"raw": text[:400]})
        except Exception:
            result = cheap

        self.cache[cache_key] = result
        return result

In [10]:
# 9. ECONOMIC AGENT

class EconomicAgent:
    @observe()
    def run(self, row: pd.Series) -> AgentResult:
        amount = safe_float(row["amount"])
        reasons = []
        score = 0.0

        if amount >= 10000:
            score += 0.40
            reasons.append("very high value")
        elif amount >= 5000:
            score += 0.25
            reasons.append("high value")
        elif amount >= 1000:
            score += 0.10
            reasons.append("moderate value")

        ttype = normalize_text(row["transaction_type"])
        if "transfer" in ttype or "bank" in ttype:
            score += 0.10
            reasons.append("transfer risk multiplier")

        return AgentResult(min(score, 1.0), reasons, {"amount": amount})

In [11]:
# 10. DRIFT AGENT

class DriftAgent:
    @observe()
    def run(self, row: pd.Series) -> AgentResult:
        desc = normalize_text(row.get("description", ""))
        score = 0.0
        reasons = []

        risky_terms = ["crypto", "voucher", "wallet", "recovery", "marketplace", "gift card"]
        hits = [t for t in risky_terms if t in desc]
        if hits:
            score += min(0.08 * len(hits), 0.25)
            reasons.append("novel description theme")

        pm = normalize_text(row.get("payment_method", ""))
        if pm in {"paypal", "googlepay", "smartwatch", "mobile device"}:
            score += 0.05
            reasons.append("digitally mediated payment")

        return AgentResult(min(score, 1.0), reasons, {"hits": hits})

In [12]:
# 11. LLM JUDGE AGENT

class LLMJudgeAgent:
    @observe()
    def run(self, row: pd.Series, evidence: Dict[str, AgentResult], session_id: str) -> AgentResult:
        if not USE_LLM:
            heuristic = min(
                1.0,
                0.45 * evidence["behavior"].score +
                0.20 * evidence["geo"].score +
                0.20 * evidence["comm"].score +
                0.10 * evidence["economic"].score +
                0.05 * evidence["drift"].score
            )
            return AgentResult(heuristic, ["heuristic judge"], {})

        packed = {
            name: {
                "score": round(res.score, 4),
                "reasons": res.reasons,
                "metadata": res.metadata
            }
            for name, res in evidence.items()
        }

        prompt = f"""
You are the final fraud adjudication agent.

Transaction:
{row.to_dict()}

Evidence:
{json.dumps(packed, ensure_ascii=False)}

Return strict JSON:
{{
  "risk": 0.0,
  "label": "fraud|legitimate",
  "confidence": 0.0,
  "reasons": ["short reason 1", "short reason 2"]
}}
""".strip()

        try:
            handler = CallbackHandler()
            resp = llm.invoke(
                [HumanMessage(content=prompt)],
                config={
                    "callbacks": [handler],
                    "metadata": {
                        "langfuse_session_id": session_id,
                        "agent_name": "LLMJudgeAgent"
                    }
                }
            )
            text = resp.content if hasattr(resp, "content") else str(resp)
            data = json.loads(text)
            risk = float(data.get("risk", 0.50))
            reasons = data.get("reasons", ["llm adjudication"])
            return AgentResult(min(max(risk, 0.0), 1.0), reasons[:6], {"raw": text[:400]})
        except Exception:
            heuristic = min(
                1.0,
                0.45 * evidence["behavior"].score +
                0.20 * evidence["geo"].score +
                0.20 * evidence["comm"].score +
                0.10 * evidence["economic"].score +
                0.05 * evidence["drift"].score
            )
            return AgentResult(heuristic, ["fallback heuristic judge"], {})

In [ ]:
# 12. ORCHESTRATOR

class FraudOrchestrator:
    def __init__(self, bundle: DatasetBundle, session_id: str):
        self.bundle = bundle
        self.session_id = session_id
        self.ctx = ContextBuilderAgent(bundle)

        self.behavior = BehaviorAgent()
        self.geo = GeoTimeAgent()
        self.comm = CommunicationAgent()
        self.economic = EconomicAgent()
        self.drift = DriftAgent()
        self.judge = LLMJudgeAgent()

        self.escalations_used = 0

    def fuse(self, res: Dict[str, AgentResult]) -> float:
        weights = {
            "behavior": 0.30,
            "geo": 0.16,
            "comm": 0.18,
            "economic": 0.16,
            "drift": 0.08,
        }
        return min(sum(weights[k] * res[k].score for k in weights), 1.0)

    @observe()
    def score_transaction(self, row: pd.Series) -> Tuple[float, Dict[str, AgentResult]]:
        messages = self.ctx.candidate_messages(row)

        results = {
            "behavior": self.behavior.run(row, self.ctx),
            "geo": self.geo.run(row, self.ctx),
            "comm": self.comm.cheap_run(messages),
            "economic": self.economic.run(row),
            "drift": self.drift.run(row),
        }

        score = self.fuse(results)

        borderline = 0.48 <= score <= 0.68
        high_value = results["economic"].score >= 0.25
        suspicious_context = results["comm"].score >= 0.32

        should_escalate = (
            USE_LLM
            and self.escalations_used < MAX_LLM_ESCALATIONS
            and (high_value or (borderline and suspicious_context))
        )

        if should_escalate:
            self.escalations_used += 1
            results["comm"] = self.comm.llm_run(messages, self.session_id)
            score = self.fuse(results)

        should_judge = (
            USE_LLM
            and self.escalations_used < MAX_LLM_ESCALATIONS
            and ((0.52 <= score <= 0.66 and suspicious_context) or high_value)
        )

        if should_judge:
            self.escalations_used += 1
            judge = self.judge.run(row, results, self.session_id)
            results["judge"] = judge
            score = 0.80 * score + 0.20 * judge.score

        return min(score, 1.0), results

    @observe()
    def run(self) -> pd.DataFrame:
        rows = []
        tx_df = self.bundle.transactions

        for row in tx_df.itertuples(index=False):
            row_s = pd.Series(row._asdict())
            risk, detail = self.score_transaction(row_s)

            all_reasons = []
            for k, v in detail.items():
                all_reasons.extend([f"{k}:{r}" for r in v.reasons])

            rows.append({
                "transaction_id": row_s["transaction_id"],
                "risk_score": risk,
                "amount": row_s["amount"],
                "behavior_score": detail["behavior"].score,
                "geo_score": detail["geo"].score,
                "comm_score": detail["comm"].score,
                "economic_score": detail["economic"].score,
                "drift_score": detail["drift"].score,
                "judge_score": detail["judge"].score if "judge" in detail else np.nan,
                "reasons": " | ".join(all_reasons)[:1200],
            })

        return pd.DataFrame(rows)

    def choose_suspicious_ids(self, scored: pd.DataFrame) -> List[str]:
        scored = scored.copy()
        scored["rank_score"] = scored["risk_score"] + 0.12 * scored["amount"].map(log1p_clip)

        q = scored["rank_score"].quantile(0.88)
        pred = scored[scored["rank_score"] >= q]["transaction_id"].tolist()

        if len(pred) == 0:
            pred = scored.nlargest(max(1, int(len(scored) * 0.03)), "rank_score")["transaction_id"].tolist()

        if len(pred) >= len(scored):
            pred = scored.nlargest(max(1, int(len(scored) * 0.20)), "rank_score")["transaction_id"].tolist()

        return pred

In [14]:
# 13. WRITE OUTPUT

def format_submission_filename(dataset_name: str) -> str:
    """
    Convert internal dataset names like:
      - Brave_New_World_train
      - The_Truman_Show_validation
      - Deus Ex train
    into submission filenames like:
      - Brave_New_World.txt
      - The_Truman_Show.txt
      - Deus_Ex.txt
    """
    safe_name = str(dataset_name).strip().replace("/", "_").replace(" ", "_")
    safe_name = re.sub(r"(?i)_(train|validation)$", "", safe_name)
    safe_name = re.sub(r"(?i)\b(train|validation)\b$", "", safe_name).strip("_")
    return f"{safe_name}.txt"


def write_submission(transaction_ids: List[str], path: str) -> None:
    """
    Write one UTF-8 encoded transaction_id per line, with no header.
    """
    cleaned_ids = [str(tid).strip() for tid in transaction_ids if str(tid).strip()]
    with open(path, "w", encoding="utf-8") as f:
        for tid in cleaned_ids:
            f.write(tid + "\n")


def main():
    print("Loading all datasets...")
    bundles = DataLoader.load_all_bundles(DATA_DIR)

    for dataset_name, bundle in bundles.items():
        print(f"\nProcessing dataset: {dataset_name}")

        session_id = generate_session_id()
        print("Langfuse session ID:", session_id)

        orchestrator = FraudOrchestrator(bundle, session_id)

        scored = orchestrator.run()

        safe_name = str(dataset_name).replace(" ", "_").replace("/", "_")
        debug_csv_path = f"./debug_{safe_name}.csv"
        output_path = f"./{format_submission_filename(dataset_name)}"

        scored.to_csv(debug_csv_path, index=False)

        suspicious_ids = orchestrator.choose_suspicious_ids(scored)
        write_submission(suspicious_ids, output_path)

        if "langfuse_client" in globals() and langfuse_client is not None:
            try:
                langfuse_client.flush()
            except Exception:
                pass

        print(f"Scored transactions: {len(scored)}")
        print(f"Flagged suspicious: {len(suspicious_ids)}")
        print(f"Submission file: {output_path}")
        print(f"Debug file: {debug_csv_path}")
        print(f"Session ID: {session_id}")


In [15]:

if __name__ == "__main__":
    main()
    

Loading all datasets...
Loading dataset: The_Truman_Show_train
Loading dataset: Deus_Ex_train
Loading dataset: Brave_New_World_train

Processing dataset: The_Truman_Show_train
Langfuse session ID: MuNA-01KPBQDGTMX3FPJ040AJFFVHC3
Scored transactions: 80
Flagged suspicious: 10
Submission file: ./The_Truman_Show.txt
Debug file: ./debug_The_Truman_Show_train.csv
Session ID: MuNA-01KPBQDGTMX3FPJ040AJFFVHC3

Processing dataset: Deus_Ex_train
Langfuse session ID: MuNA-01KPBQDNWKMA3FV3GF4WCWGXT7


Queue full, dropping Span.
Failed to export span batch due to timeout, max retries or shutdown.
Queue full, dropping Span.
Failed to export span batch due to timeout, max retries or shutdown.
Failed to export span batch due to timeout, max retries or shutdown.
Queue full, dropping Span.
Failed to export span batch due to timeout, max retries or shutdown.
Failed to export span batch due to timeout, max retries or shutdown.
Failed to export span batch due to timeout, max retries or shutdown.
Failed to export span batch due to timeout, max retries or shutdown.
Failed to export span batch due to timeout, max retries or shutdown.
Failed to export span batch due to timeout, max retries or shutdown.


Scored transactions: 2017
Flagged suspicious: 242
Submission file: ./Deus_Ex.txt
Debug file: ./debug_Deus_Ex_train.csv
Session ID: MuNA-01KPBQDNWKMA3FV3GF4WCWGXT7

Processing dataset: Brave_New_World_train
Langfuse session ID: MuNA-01KPBQGM15ZGPQV1YAFTXN7SPM


Queue full, dropping Span.
Failed to export span batch due to timeout, max retries or shutdown.


Scored transactions: 522
Flagged suspicious: 63
Submission file: ./Brave_New_World.txt
Debug file: ./debug_Brave_New_World_train.csv
Session ID: MuNA-01KPBQGM15ZGPQV1YAFTXN7SPM
